<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/AutonomousAgent_multivendor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. INSTALL DEPENDENCIES
!pip install -q git+https://github.com/KarAnalytics/llm_cascade.git

  Preparing metadata (setup.py) ... done


**IGNORE THE ABOVE ERRORS AND PROCEED** — they are specific to Google Colab's preinstalled packages and the code will still execute correctly.

# Autonomous Agent: Business Idea Validator (Multi-Vendor)

This notebook extends the base autonomous workflow by assigning **a different free-tier LLM vendor to each phase**. Instead of one shared cascade instance, each agent phase has its own `LLMCascade` instance pinned to a specific provider — demonstrating that autonomous workflows can mix and match models from different vendors within the same run.

**Phase → Vendor mapping (defaults):**

| Phase | Role | Default vendor | Why |
|---|---|---|---|
| Plan | Strategic planner | Gemini | Strong at structured decomposition |
| Execute | Business analyst | Groq | Fast, factual, low latency |
| Synthesize | Senior executive | Cohere | Strong summarisation / reasoning |
| Reflect | Skeptical critic | HuggingFace | Independent perspective |

Each vendor is loaded from your existing API keys — no extra setup required beyond what `llm_cascade` already supports.

### How is this different from `AutonomousAgent_BusinessValidator.ipynb`?

| | Single-vendor autonomous | **Multi-vendor autonomous (this)** |
|---|---|---|
| LLM instances | 1 shared cascade | **4 separate per-phase instances** |
| Provider per phase | Whichever responds first | **Pinned to a specific vendor** |
| Fallback | Yes — across all configured keys | **Optional — can keep cascade fallback per phase** |
| Teaching point | Autonomous workflow basics | **Different LLMs can specialise in different cognitive tasks** |

## Imports and Setup

Each phase gets its own `LLMCascade` instance pinned to a specific provider. The `pin()` helper accepts one or more provider names; if you pass multiple names the phase still gets cascade-style fallback but only across those named vendors.

Edit the vendor names passed to `pin()` to swap models — all available names are listed by `get_cascade()` at startup.

In [2]:
from llm_cascade import get_cascade
from llm_cascade.providers import LLMCascade, PROVIDERS

# Show all configured providers so students know what's available
print('All configured providers:')
_ = get_cascade()


def pin(*provider_names):
    """Return an LLMCascade restricted to the named provider(s).
    Falls back across names in the order given, just like the full cascade.
    If a named provider has no API key it is silently skipped."""
    selected = [p for p in PROVIDERS if p['name'] in provider_names]
    if not selected:
        raise ValueError(f'None of {provider_names} found in PROVIDERS. '
                         f'Available: {[p["name"] for p in PROVIDERS]}')
    return LLMCascade(providers=selected, verbose=True)


# --- Assign one vendor per phase ---
# Change any of these to any provider name shown above.
# You can also pass multiple names for per-phase fallback, e.g. pin('Gemini', 'Groq')
planner_llm     = pin('Gemini')        # strategic decomposition
executor_llm    = pin('Groq')          # fast, factual analyst
synthesizer_llm = pin('Cohere')        # summarisation & recommendation
reflector_llm   = pin('HuggingFace')   # independent skeptic

print()
print('Phase → vendor assignments:')
for label, instance in [('Planner', planner_llm), ('Executor', executor_llm),
                         ('Synthesizer', synthesizer_llm), ('Reflector', reflector_llm)]:
    names = ', '.join(p['name'] for p in instance.available) or 'NONE (key missing!)'
    print(f'  {label:<12} -> {names}')

All configured providers:
LLM Cascade - available providers:
  + Gemini           model=gemini-2.5-flash
  + Ollama           model=kimi-k2.5:cloud
  + Groq             model=llama-3.3-70b-versatile
  + HuggingFace      model=meta-llama/Llama-3.3-70B-Instruct
  + Cohere           model=command-a-03-2025
  + OpenRouter       model=meta-llama/llama-3.3-70b-instruct:free
  + OpenAI           model=gpt-4o-mini
Not configured (skipped):
  - Grok (xAI)       (set XAI_API_KEY)
LLM Cascade - available providers:
  + Gemini           model=gemini-2.5-flash
LLM Cascade - available providers:
  + Groq             model=llama-3.3-70b-versatile
LLM Cascade - available providers:
  + Cohere           model=command-a-03-2025
LLM Cascade - available providers:
  + HuggingFace      model=meta-llama/Llama-3.3-70B-Instruct

Phase → vendor assignments:
  Planner      -> Gemini
  Executor     -> Groq
  Synthesizer  -> Cohere
  Reflector    -> HuggingFace


## The Four Phases of the Autonomous Workflow

The logic is identical to the single-vendor version. The only change is that each function calls its own dedicated LLM instance instead of the shared `llm` object.

1. **Plan** — `planner_llm` (Gemini) generates 4–5 research questions
2. **Execute** — `executor_llm` (Groq) answers each question, accumulating context
3. **Synthesize** — `synthesizer_llm` (Cohere) combines findings into a recommendation
4. **Reflect** — `reflector_llm` (HuggingFace) critiques the recommendation and scores it

In [3]:
# Phase 1: Planner -- decides WHAT needs to be researched
PLANNER_SYSTEM = (
    'You are a strategic business planner. Given a high-level goal, break it down '
    'into the 4-5 most critical questions that must be answered to achieve it. '
    'Output ONLY a numbered list of questions. No preamble, no conclusion.'
)


def plan(goal):
    '''Phase 1: generate a research plan as a list of questions (uses planner_llm).'''
    prompt = f'Goal: {goal}' + chr(10) + chr(10) + 'List the 4-5 most critical questions to answer.'
    response = planner_llm.generate(prompt, system_prompt=PLANNER_SYSTEM)

    # Parse the numbered list into a Python list of strings
    steps = []
    for line in response.text.split(chr(10)):
        line = line.strip()
        if not line:
            continue
        # Strip leading '1.', '1)', '-', '*', etc.
        cleaned = line.lstrip('0123456789.)- *').strip()
        if cleaned and len(cleaned) > 5:
            steps.append(cleaned)
    return steps


print('Phase 1 (Planner) ready.')

Phase 1 (Planner) ready.


In [4]:
# Phase 2: Executor -- answers ONE research question at a time
EXECUTOR_SYSTEM = (
    'You are a business analyst. Answer the question concisely (2-4 sentences) '
    'with concrete reasoning. Use the prior context if helpful, but stay focused '
    'on the current question.'
)


def execute_step(question, prior_context):
    '''Phase 2: answer one research question with access to prior answers (uses executor_llm).'''
    prompt = 'Prior context:' + chr(10) + prior_context + chr(10) + chr(10) + 'Question: ' + question
    response = executor_llm.generate(prompt, system_prompt=EXECUTOR_SYSTEM)
    return response.text


print('Phase 2 (Executor) ready.')

Phase 2 (Executor) ready.


In [5]:
# Phase 3: Synthesizer -- combines findings into a recommendation
SYNTHESIZER_SYSTEM = (
    'You are a senior executive. Given research findings, synthesize them into a '
    'clear, decisive recommendation. State the recommendation (go / no-go / conditional), '
    'followed by 3 bullet points of key reasoning.'
)


def synthesize(goal, research_results):
    '''Phase 3: combine all research answers into a final recommendation (uses synthesizer_llm).'''
    findings = chr(10).join([f'Q: {q}' + chr(10) + f'A: {a}' for q, a in research_results])
    prompt = (
        'Goal: ' + goal + chr(10) + chr(10)
        + 'Research findings:' + chr(10) + findings + chr(10) + chr(10)
        + 'Synthesize into a final recommendation.'
    )
    response = synthesizer_llm.generate(prompt, system_prompt=SYNTHESIZER_SYSTEM)
    return response.text


print('Phase 3 (Synthesizer) ready.')

Phase 3 (Synthesizer) ready.


In [6]:
# Phase 4: Critic -- self-reflects on the recommendation
CRITIC_SYSTEM = (
    'You are a skeptical business critic. Given a recommendation, identify '
    'what is weak or missing, what is strong, and assign a confidence level '
    'from 1 (very weak) to 10 (very strong). Be concise.'
)


def reflect(recommendation):
    '''Phase 4: critique the recommendation and assign confidence (uses reflector_llm).'''
    prompt = 'Recommendation to critique:' + chr(10) + recommendation
    response = reflector_llm.generate(prompt, system_prompt=CRITIC_SYSTEM)
    return response.text


print('Phase 4 (Critic) ready.')

Phase 4 (Critic) ready.


## The Autonomous Loop

Same logic as the single-vendor notebook, but now each phase label also shows **which vendor responded**. Watch the `[Response from ...]` lines to see the four different LLMs handing off to each other.

Total LLM calls per run: **1 (plan) + 4–5 (execute) + 1 (synthesize) + 1 (reflect) = 7–8 calls**, each routed to a different provider.

In [7]:
def run_autonomous_workflow(goal, verbose=True):
    '''The full autonomous workflow: Plan -> Execute -> Synthesize -> Reflect.
    Each phase uses a different LLM vendor.'''
    if verbose:
        print('=' * 70)
        print('GOAL:', goal)
        print('=' * 70)

    # --- Phase 1: Plan (planner_llm) ---
    if verbose:
        print(chr(10) + '[PHASE 1 - PLANNING  |  vendor: ' +
              ', '.join(p['name'] for p in planner_llm.available) + ']')
    steps = plan(goal)
    if verbose:
        print(f'Agent decided to research {len(steps)} questions:')
        for i, s in enumerate(steps, 1):
            print(f'  {i}. {s}')

    # --- Phase 2: Execute each step (executor_llm) ---
    if verbose:
        print(chr(10) + '[PHASE 2 - EXECUTION  |  vendor: ' +
              ', '.join(p['name'] for p in executor_llm.available) + ']')
    results = []
    prior_context = ''
    for i, step in enumerate(steps, 1):
        if verbose:
            print(chr(10) + f'Step {i}: {step}')
        answer = execute_step(step, prior_context)
        results.append((step, answer))
        prior_context += chr(10) + f'Q: {step}' + chr(10) + f'A: {answer}'
        if verbose:
            print(f'  -> {answer}')

    # --- Phase 3: Synthesize (synthesizer_llm) ---
    if verbose:
        print(chr(10) + '[PHASE 3 - SYNTHESIS  |  vendor: ' +
              ', '.join(p['name'] for p in synthesizer_llm.available) + ']')
    recommendation = synthesize(goal, results)
    if verbose:
        print(recommendation)

    # --- Phase 4: Reflect (reflector_llm) ---
    if verbose:
        print(chr(10) + '[PHASE 4 - SELF-CRITIQUE  |  vendor: ' +
              ', '.join(p['name'] for p in reflector_llm.available) + ']')
    critique = reflect(recommendation)
    if verbose:
        print(critique)

    return {
        'goal': goal,
        'plan': steps,
        'research': results,
        'recommendation': recommendation,
        'critique': critique,
    }


print('Autonomous multi-vendor workflow function ready.')

Autonomous multi-vendor workflow function ready.


## Run the Agent

Edit `business_idea` and run the cell. Watch the vendor tag on each phase header to confirm that different LLMs are handling different stages of the reasoning chain.

In [8]:
business_idea = 'An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.'

# business_idea = 'A subscription meal kit service for busy professionals that uses local ingredients.'
# business_idea = 'A mobile game that teaches kids basic accounting through story-based puzzles.'

result = run_autonomous_workflow(business_idea)

GOAL: An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.

[PHASE 1 - PLANNING  |  vendor: Gemini]
  [Response from Gemini / gemini-2.5-flash]
Agent decided to research 5 questions:
  1. What data sources are reliably available for off-campus housing listings, historical rent prices, and relevant neighborhood characteristics (e.g., crime rates, public transport, amenities) near target universities?
  2. How accurate can an AI model predict rent trends for specific apartment types and locations, considering the volatility and unique factors of student housing markets?
  3. What are the key features and user experience elements necessary to make the app intuitive, trustworthy, and genuinely helpful for college students on a budget?
  4. What is the most effective go-to-market strategy to reach and acquire a critical mass of college student users at target universities?
  5. What legal and ethical considerations (e.g., data privacy, 

## Inspect the Full Trace

The `result` dict holds everything the agent produced. Each answer in `result['research']` was generated by `executor_llm`; the recommendation by `synthesizer_llm`; and the critique by `reflector_llm` — three different LLMs contributing to one coherent report.

In [9]:
print('=' * 70)
print('FULL TRACE')
print('=' * 70)
print(chr(10) + 'GOAL:', result['goal'])
print(chr(10) + 'PLAN (what the agent decided to research):')
for i, step in enumerate(result['plan'], 1):
    print(f'  {i}. {step}')
print(chr(10) + 'FINAL RECOMMENDATION:')
print(result['recommendation'])
print(chr(10) + 'SELF-CRITIQUE:')
print(result['critique'])

FULL TRACE

GOAL: An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.

PLAN (what the agent decided to research):
  1. What data sources are reliably available for off-campus housing listings, historical rent prices, and relevant neighborhood characteristics (e.g., crime rates, public transport, amenities) near target universities?
  2. How accurate can an AI model predict rent trends for specific apartment types and locations, considering the volatility and unique factors of student housing markets?
  3. What are the key features and user experience elements necessary to make the app intuitive, trustworthy, and genuinely helpful for college students on a budget?
  4. What is the most effective go-to-market strategy to reach and acquire a critical mass of college student users at target universities?
  5. What legal and ethical considerations (e.g., data privacy, fair housing practices, algorithmic bias) must be addressed to build

## Key Takeaways

**What's new vs. the single-vendor version:**
- Each phase is served by a **different LLM from a different vendor** — they communicate by passing text, not by sharing state
- The `pin()` helper lets you **assign any provider from `PROVIDERS`** to any phase in one line
- You can pass multiple names to `pin()` for per-phase fallback: `pin('Gemini', 'Groq')` tries Gemini first, then Groq — useful if one key is missing
- The vendor name appears in each phase header so you can verify the routing in the output

**Why use different vendors per phase?**
- Some models excel at **decomposition** (planning), others at **fast factual retrieval** (execution), others at **synthesis** or **critical evaluation**
- It reduces dependence on a single provider's quota or availability
- It's the same principle behind multi-agent frameworks like CrewAI or AutoGen, but without the orchestration overhead

**Exercises:**
- Swap the vendor assignments and compare the quality of the output — does a different planner generate better research questions?
- Pass two vendors to `pin()` (e.g., `pin('Cohere', 'Groq')`) and pull the API key for the first one to test per-phase fallback
- Add a fifth phase (e.g., a marketing writer) using whichever vendor has remaining quota
- Modify `run_autonomous_workflow` so that if the critic's confidence score is below 7, the critique is fed back to `planner_llm` for a revised plan